# Prop 2.5 Override vote data

Waltham is lucky enough not to have had an override vote as of 2026, but we're dependent on commerical taxes supplied by corporate offices that are less occupied now given the hybrid work model.
As a result, this may be our future.

Source:
https://dls-gw.dor.state.ma.us/reports/rdPage.aspx?rdReport=Votes.Prop2_5.OverrideUnderride

In [6]:
import re
import requests
import pandas as pd
import plotly.graph_objects as go

from plotly.subplots import make_subplots
from pathlib import Path
from datetime import datetime

CURRENT_YEAR = 2026


# Fetch data

The state doesn't have an API to grab this as far as I know, so we have to manipulate the page to maximize the amount of data returned, and we get an Excel file out.

In [ ]:

# Fetch the page to read all current checkbox values (municipalities, years, departments, etc.)
BASE_URL = "https://dls-gw.dor.state.ma.us/reports/rdPage.aspx"
FILEPATH = Path("investigations/city_budget") / Path("override_votes.xlsx")
session = requests.Session()
r = session.get(BASE_URL, params={"rdReport": "Votes.Prop2_5.OverrideUnderride"})
r.raise_for_status()

# Collect all checked values per field name
post_data = []
for m in re.finditer(r'<input[^>]+type="checkbox"[^>]+name="([^"]+)"[^>]+value="([^"]+)"', r.text, re.IGNORECASE):
    post_data.append((m.group(1), m.group(2)))

# Include hidden fields required by the report engine
for m in re.finditer(r'<INPUT TYPE="HIDDEN"[^>]+NAME="([^"]+)"[^>]*>', r.text, re.IGNORECASE):
    val_m = re.search(r'VALUE="([^"]*)"', m.group(), re.IGNORECASE)
    post_data.append((m.group(1), val_m.group(1) if val_m else ""))

# POST to the Excel export endpoint with all filters selected
export_params = {
    "rdReport": "Votes.Prop2_5.OverrideUnderride",
    "rdReportFormat": "NativeExcel",
    "rdExportTableID": "tblProp2_5Votes",
    "rdExportFilename": "OverrideUnderrideVotes",
    "rdShowGridlines": "True",
    "rdExcelOutputFormat": "Excel2007",
}
resp = session.post(BASE_URL, params=export_params, data=post_data)
resp.raise_for_status()

with open(FILEPATH, "wb") as f:
    f.write(resp.content)

print(f"Downloaded {len(resp.content):,} bytes")
df = pd.read_excel(FILEPATH)


In [ ]:
# Inflation-adjust Amount to CURRENT_YEAR dollars at 2.5% per year
df["Amount_adj"] = df["Amount"] * (1.025 ** (CURRENT_YEAR - df["Fiscal Year"]))

overrides = df[df["Vote Type"] == "Override"].copy()

# --- Who has the most / least override votes ---
vote_counts = (
    overrides.groupby("Municipality")
    .size()
    .rename("votes")
    .sort_values(ascending=False)
)
print("Most override votes")
print(vote_counts.head(10).to_string())
print()
print("Least override votes (bottom 10)")
print(vote_counts.tail(10).to_string())

# --- Who fails / succeeds most ---
win_rate = (
    overrides.groupby("Municipality")["Win / Loss"]
    .apply(lambda s: (s == "WIN").mean())
    .rename("win_rate")
)
counts = vote_counts.rename("n_votes")
win_summary = pd.concat([counts, win_rate], axis=1).dropna()

print("\nHighest win rate (min 5 votes)")
high_win = win_summary[win_summary["n_votes"] >= 5].sort_values("win_rate", ascending=False)
print(high_win.head(10).to_string())

print("\nLowest win rate (min 5 votes)")
print(high_win.tail(10).sort_values("win_rate").to_string())

# --- Highest total inflation-adjusted override amount ---
total_adj = (
    overrides[overrides["Win / Loss"] == "WIN"]
    .groupby("Municipality")["Amount_adj"]
    .sum()
    .sort_values(ascending=False)
)
print("\nHighest total passed override amount (inflation-adjusted to {})".format(CURRENT_YEAR))
print(total_adj.head(10).apply(lambda x: f"${x:,.0f}").to_string())

# --- Which departments are most affected ---
dept = (
    overrides.groupby("Department")
    .agg(
        n_votes=("Amount", "count"),
        win_rate=("Win / Loss", lambda s: (s == "WIN").mean()),
        total_adj=("Amount_adj", "sum"),
    )
    .sort_values("n_votes", ascending=False)
)
dept["total_adj"] = dept["total_adj"].apply(lambda x: f"${x:,.0f}")
dept["win_rate"] = dept["win_rate"].apply(lambda x: f"{x:.0%}")
print("\nDepartments by number of override votes")
print(dept.to_string())

Most override votes
Municipality
Chatham         94
West Tisbury    91
Tisbury         90
Wellfleet       84
Holland         83
Eastham         78
Orange          76
Harwich         75
Aquinnah        71
Boxford         70

Least override votes (bottom 10)
Municipality
Plainfield     1
Plymouth       1
Chelsea        1
Tyringham      1
Wakefield      1
Stockbridge    1
Stoughton      1
Saugus         1
Swansea        1
Barre          1

Highest win rate (min 5 votes)
              n_votes  win_rate
Municipality                   
Milton             10  1.000000
Weston              9  1.000000
Rockport            8  1.000000
Heath               5  1.000000
Lynnfield           5  1.000000
Brookline           6  1.000000
Sherborn           11  1.000000
Orleans            36  0.972222
Chilmark           63  0.952381
Brewster           20  0.950000

Lowest win rate (min 5 votes)
              n_votes  win_rate
Municipality                   
Revere             14       0.0
South Hadley     

# Is this a recent phenomenon?

According to the Strong Towns thesis, suburbanization is generally not fiscally possible in the long run, so we'd expect communities to struggle to get their budgets right as they age.

We'll bin data by decade and look at statewide trends (number of override votes and win rates), then look at the top 30 communities statewide.

In [ ]:
RECENT_CUTOFF = 2020

overrides = df[df["Vote Type"] == "Override"].copy()
overrides["decade"] = (overrides["Fiscal Year"] // 10) * 10

# ── 1. Annual vote count + win rate ──────────────────────────────────────────
by_year = overrides.groupby("Fiscal Year").agg(
    n_votes=("Amount", "count"),
    win_rate=("Win / Loss", lambda s: (s == "WIN").mean()),
)

fig1 = make_subplots(specs=[[{"secondary_y": True}]])
fig1.add_bar(x=by_year.index, y=by_year["n_votes"], name="Override votes", opacity=0.6, secondary_y=False)
fig1.add_scatter(x=by_year.index, y=by_year["win_rate"], name="Win rate", mode="lines+markers", secondary_y=True)
fig1.update_layout(title="Override votes per year — statewide", xaxis_title="Fiscal Year")
fig1.update_yaxes(title_text="Number of votes", secondary_y=False)
fig1.update_yaxes(title_text="Win rate", tickformat=".0%", secondary_y=True)
fig1.show()

# ── 2. Community activity by decade — are top communities a 90s artefact? ───
DECADES = [1990, 2000, 2010, 2020]
DECADE_LABELS = ["1990s", "2000s", "2010s", "2020s"]

# Identify communities active in 2020s vs only historically
recent_active = set(overrides.loc[overrides["Fiscal Year"] >= RECENT_CUTOFF, "Municipality"].unique())
historic_only = set(overrides.loc[overrides["Fiscal Year"] < RECENT_CUTOFF, "Municipality"].unique()) - recent_active

top_by_total = overrides.groupby("Municipality").size().nlargest(30).index
decade_pivot = (
    overrides[overrides["Municipality"].isin(top_by_total)]
    .groupby(["Municipality", "decade"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=DECADES, fill_value=0)
)
decade_pivot.columns = DECADE_LABELS
decade_pivot["recent_flag"] = decade_pivot.index.isin(recent_active)
decade_pivot = decade_pivot.sort_values("1990s", ascending=True)

fig2 = go.Figure()
colors = {"1990s": "#d62728", "2000s": "#ff7f0e", "2010s": "#1f77b4", "2020s": "#2ca02c"}
for col in DECADE_LABELS:
    fig2.add_bar(
        y=decade_pivot.index,
        x=decade_pivot[col],
        name=col,
        orientation="h",
        marker_color=colors[col],
    )
fig2.update_layout(
    barmode="stack",
    title="Override vote activity by decade — top 30 communities",
    xaxis_title="Number of override votes",
    height=700,
    legend_title="Decade",
)
fig2.show()

# ── 3. Communities that are genuinely recent (first vote after 2010) ─────────
first_vote = overrides.groupby("Municipality")["Fiscal Year"].min()
new_communities = first_vote[first_vote >= 2010].sort_values()
print(f"Communities whose first-ever override vote came in 2010 or later ({len(new_communities)} total):")
print(new_communities.to_string())

# ── 4. Win rate trend for historically dominant vs recently active communities ─
top_historic = ["Chatham", "Tisbury", "West Tisbury", "Holland", "Orange"]
top_recent = overrides[overrides["Fiscal Year"] >= RECENT_CUTOFF].groupby("Municipality").size().nlargest(5).index.tolist()

fig3 = go.Figure()
for muni in top_historic + top_recent:
    sub = (
        overrides[overrides["Municipality"] == muni]
        .groupby("Fiscal Year")["Win / Loss"]
        .apply(lambda s: (s == "WIN").mean())
        .reset_index()
    )
    is_recent = muni in top_recent
    fig3.add_scatter(
        x=sub["Fiscal Year"],
        y=sub["Win / Loss"],
        name=muni + (" (recent)" if is_recent else " (historic)"),
        mode="lines+markers",
        line=dict(dash="dash" if is_recent else "solid"),
    )
fig3.update_layout(
    title="Annual win rate: historically dominant vs currently active communities",
    xaxis_title="Fiscal Year",
    yaxis_tickformat=".0%",
    yaxis_title="Win rate",
)
fig3.show()

# ── Summary ───────────────────────────────────────────────────────────────────
decade_summary = overrides.groupby("decade").agg(
    n_votes=("Amount", "count"),
    win_rate=("Win / Loss", lambda s: (s == "WIN").mean()),
    n_communities=("Municipality", "nunique"),
)
decade_summary.index = DECADE_LABELS
decade_summary["win_rate"] = decade_summary["win_rate"].apply(lambda x: f"{x:.0%}")
print("\nDecade summary:")
print(decade_summary.to_string())

Communities whose first-ever override vote came in 2010 or later (10 total):
Municipality
North Adams           2012
Millville             2012
Plainfield            2014
Grafton               2015
North Attleborough    2019
Norwood               2020
West Brookfield       2025
Medford               2025
Barre                 2026
Malden                2027



Decade summary:
       n_votes win_rate  n_communities
1990s     2767      33%            277
2000s     1174      51%            205
2010s      402      54%            127
2020s      368      62%            133


Generally the story that appears above is that there are fewer override votes recently vs. in the 1990s. Perhaps the story here is that Prop 2.5 is forcing communities to be careful with their budgets generally, so the overrides are needed less often.

# Which departments are most affected?

In [ ]:
CURRENT_YEAR = 2026
DECADE_LABELS = ["1990s", "2000s", "2010s", "2020s"]

df = pd.read_excel("investigations/city_budget/override_votes.xlsx")
overrides = df[df["Vote Type"] == "Override"].copy()
overrides["Amount_adj"] = overrides["Amount"] * (1.025 ** (CURRENT_YEAR - overrides["Fiscal Year"]))
overrides["decade"] = (overrides["Fiscal Year"] // 10) * 10

# ── 1. Overview: votes, win rate, and dollars by department ──────────────────
dept = (
    overrides.groupby("Department")
    .agg(
        n_votes=("Amount", "count"),
        win_rate=("Win / Loss", lambda s: (s == "WIN").mean()),
        amount_adj=("Amount_adj", "sum"),
    )
    .sort_values("n_votes", ascending=True)
)

fig1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Override votes by department", "Inflation-adjusted amount sought (2026 $)"),
)
fig1.add_bar(
    y=dept.index, x=dept["n_votes"],
    orientation="h", name="Votes",
    marker_color=[f"rgba(31,119,180,{w:.2f})" for w in dept["win_rate"]],
    customdata=dept["win_rate"],
    hovertemplate="%{y}<br>Votes: %{x}<br>Win rate: %{customdata:.0%}<extra></extra>",
    row=1, col=1,
)
fig1.add_bar(
    y=dept.index, x=dept["amount_adj"] / 1e9,
    orientation="h", name="Amount ($B)",
    marker_color="steelblue",
    hovertemplate="%{y}<br>$%{x:.2f}B<extra></extra>",
    row=1, col=2,
)
fig1.update_layout(
    title="Department overview — bar color intensity reflects win rate",
    height=450,
    showlegend=False,
)
fig1.update_xaxes(title_text="Number of votes", row=1, col=1)
fig1.update_xaxes(title_text="Billions (2026 $)", row=1, col=2)
fig1.show()

# ── 2. Department mix over time ───────────────────────────────────────────────
decade_dept = (
    overrides.groupby(["decade", "Department"])
    .size()
    .unstack(fill_value=0)
)
decade_dept.index = DECADE_LABELS

# Normalise to share of votes each decade
decade_dept_pct = decade_dept.div(decade_dept.sum(axis=1), axis=0)

fig2 = go.Figure()
for dept_name in decade_dept_pct.columns:
    fig2.add_bar(
        x=decade_dept_pct.index,
        y=decade_dept_pct[dept_name],
        name=dept_name,
    )
fig2.update_layout(
    barmode="stack",
    title="Share of override votes by department per decade",
    yaxis_tickformat=".0%",
    yaxis_title="Share of votes",
    legend_title="Department",
)
fig2.show()

# ── 3. Win rate by department over time ──────────────────────────────────────
decade_win = (
    overrides.groupby(["decade", "Department"])["Win / Loss"]
    .apply(lambda s: (s == "WIN").mean())
    .unstack()
)
decade_win.index = DECADE_LABELS

fig3 = go.Figure()
for dept_name in decade_win.columns:
    fig3.add_scatter(
        x=decade_win.index,
        y=decade_win[dept_name],
        name=dept_name,
        mode="lines+markers",
    )
fig3.update_layout(
    title="Win rate by department per decade",
    yaxis_tickformat=".0%",
    yaxis_title="Win rate",
    legend_title="Department",
)
fig3.show()

# ── 4. Print summary table ────────────────────────────────────────────────────
summary = dept.sort_values("n_votes", ascending=False).copy()
summary["win_rate"] = summary["win_rate"].apply(lambda x: f"{x:.0%}")
summary["amount_adj"] = summary["amount_adj"].apply(lambda x: f"${x/1e6:,.0f}M")
summary.columns = ["Votes", "Win rate", "Amount (2026 $M)"]
print(summary.to_string())

                            Votes Win rate Amount (2026 $M)
Department                                                 
SCHOOL                       1290      49%          $1,042M
GENERAL OPERATING            1143      43%          $1,469M
PUBLIC SAFETY                 612      41%            $147M
PUBLIC WORKS & FACILITIES     603      35%            $151M
GENERAL GOVERNMENT            547      30%            $462M
CULTURE AND RECREATION        331      40%             $33M
HEALTH AND HUMAN SERVICE      127      51%             $10M
EMPLOYEE BENEFITS              32      44%              $8M
FUNDS                          23      30%              $7M
PRE-PROPOSITION 2 1/2 DEBT      3       0%             $13M
